In [ ]:
import jax.numpy as jnp
import numpy as np
from fft import *
import jax
from symmetry import *
from phase_retrieval import *
from obj import *
from noise import *
from mle import *
from plotting import plot
import device
from jax import lax
from pies import *
from pies import *
from symmetry import choose_representative
import matplotlib.pyplot as plt
from ptychography import *
from imaging import make_circular_mask

jax.config.update("jax_enable_x64", True)
cdtype = jnp.array(1j).dtype
gpu_idx, free_mb, gpu_count = device.setup_device(device_id=0)

In [ ]:
N_crop = 255
N_pad = 0

real_object = False

# Ensure that cameraman is nowhere exactly zero to avoid possible singular behavior (for now)
nonzero_cameraman = cameraman() * 0.9 + 0.1

if real_object:
    # Construct object with cameraman in intensity
    x_obj, obj_support = crop_and_pad(nonzero_cameraman, N_crop=N_crop, N_pad=N_pad)
else:
    # Construct object with cameraman in intensity and mandrill in phase
    x_cameraman = nonzero_cameraman
    x_mandrill = as_phase_field(mandrill() ** 2 - 0.5)
    x_obj, obj_support = crop_and_pad(
        x_cameraman * x_mandrill, N_crop=N_crop, N_pad=N_pad
    )

x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = create_physical_axes(
    x_obj
)

N_compare_crop = N_crop // 2


print(r_extent, k_extent)
# Define linear operator

NA_detector = 1.0

F_pupil = make_circular_mask(k, k_0 * NA_detector)


N_compare_crop = N_crop // 2

print(r_extent, k_extent)
# Define linear operator
A = jit(lambda x: F_pupil * fft2_shifted(x))
A_dagger = jit(lambda x: ifft2_shifted(F_pupil * x))
yhat = A(x_obj)

plot_intensity_phase(x_obj, extent=r_extent)
plot_intensity_phase(yhat, extent=k_extent, use_log=True)

In [ ]:
from spotgen import *

min_x, max_x, min_y, max_y = r_extent
# Fraction of r_extent to fill with the multispot array
frac_x = 0.25
NA = 0.5

# 1. Multi-spot real-space ptychography with slight 2D scanning
Slight translations of the multi-spot pattern using a galvo-mirror should be possible. 

In [ ]:
L_x, L_y = 24, 24
NA = 0.5
R_opt = 1 / (2 * NA)  # lambda
resolution = R_opt
print(R_opt)
frac_x = 0.3
frac_y = 0.3
array_bounds = min_x * frac_x, max_x * frac_x, min_y * frac_y, max_y * frac_y
p_x = (array_bounds[1] - array_bounds[0]) / (L_x - 1)
p_y = (array_bounds[3] - array_bounds[2]) / (L_y - 1)
print(p_x, p_y, resolution)
# gamma = np.arccos(L_x * R / (2 * p))
# print(L_x * R / (2 * p))
angle_gamma = np.arctan(1 / L_x)

In [ ]:
M_x = 4
M_y = 4
M = M_x * M_y
base_key = jax.random.PRNGKey(1776)

frac_x = 0.7
frac_y = 0.7
array_bounds = min_x * frac_x, max_x * frac_x, min_y * frac_y, max_y * frac_y
p_x = (array_bounds[1] - array_bounds[0]) / (L_x - 1)
p_y = (array_bounds[3] - array_bounds[2]) / (L_y - 1)

# Use unit cell as step size
r = (array_bounds[1] - array_bounds[0]) / (L_x - 1)
delta_r = r * 3.0
# delta_r = r * 0.01
deltas = jax.random.uniform(base_key, shape=(M, 2), minval=-delta_r, maxval=delta_r)

R_probes = deltas
R_probes_org = deltas


f_ms = jit(
    lambda R_x, R_y: create_multispot_array_product(
        NA,
        array_bounds[0] + R_x,
        array_bounds[1] + R_x,
        array_bounds[2] + R_y,
        array_bounds[3] + R_y,
        L_x,
        L_y,
        K_X,
        K_Y,
        k,
        k_0,
    )
)

N_photons = 1
probes = create_probes(R_probes, f_ms, N_photons, cdtype)

ys = A(x_obj * probes)
plot_intensity_phase(ys[-1, :], use_log=True)

probed = analyse_probe(probes, x_obj, r_extent)
plot_R_probes(R_probes_org, r)

In [ ]:
x_est = pie(
    ys, probes, gamma=0.75, N_iter=400, obj_support=obj_support, A=A, A_dagger=A_dagger
)
compare_estimate(x_est * probed, x_obj * probed, N_compare_crop=N_crop)

In [ ]:
x_est = ppie(
    ys, probes, gamma=0.75, N_iter=400, obj_support=obj_support, A=A, A_dagger=A_dagger
)
compare_estimate(x_est * probed, x_obj * probed, N_compare_crop=N_crop)

# 2. Experiment: optimize uniformity of original random probe positions

In [ ]:
dx = x[1] - x[0]
dy = y[1] - y[0]
W_pixels = (array_bounds[1] - array_bounds[0]) / dx
H_pixels = (array_bounds[3] - array_bounds[2]) / dy
goal_field_density = jnp.sqrt(M * N_photons / (W_pixels * H_pixels))
goal_field_density

In [ ]:
N_mask = 100
if (N_crop - N_mask) % 2 == 1:
    N_mask += 1
mask = pad_image(jnp.ones((N_mask, N_mask)), int((N_crop - N_mask) / 2))
# x_obj = pad_image(cropped_x, N_pad)
# obj_support = pad_image(obj_support_unpadded, N_pad)


def error_2D_from_probes(R_probes):
    probes = create_probes(R_probes, f_ms, N_photons, cdtype)
    probe_sum = jnp.sum(jnp.abs(probes) ** 2, axis=0)
    error_from_uniform = (probe_sum / goal_field_density**2 - 1) * mask
    return error_from_uniform


error_from_uniform = error_2D_from_probes(R_probes)
plot(error_from_uniform)
jnp.mean(jnp.abs(error_from_uniform))

In [ ]:
import optax
from jax import grad


def f_cost(R_probes):
    return jnp.sum(jnp.abs(error_2D_from_probes(R_probes)) ** 2)


g_error = jit(grad(f_cost))


@jit
def optimize_for_uniformity(R_probes, N_iter=100, lr=1e-2):
    """Optimize probe positions to minimize non-uniformity photon counts in middle section."""
    optimizer = optax.adam(lr)
    opt_state = optimizer.init(R_probes)

    def body(_, x):
        R_probes, opt_state = x
        grads = g_error(R_probes)
        updates, opt_state = optimizer.update(grads, opt_state)
        R_probes = optax.apply_updates(R_probes, updates)
        return (R_probes, opt_state)

    R_probes, opt_state = lax.fori_loop(0, N_iter, body, (R_probes, opt_state))
    return R_probes

In [ ]:
R_probes = optimize_for_uniformity(R_probes)

In [ ]:
error_from_uniform = error_2D_from_probes(R_probes)
plot(error_from_uniform)
jnp.mean(jnp.abs(error_from_uniform))
probes = create_probes(R_probes, f_ms, N_photons, cdtype)
probed = analyse_probe(probes, x_obj, r_extent)

In [ ]:
plot_R_probes(R_probes, r)
plot_R_probes(R_probes_org, r)

In [ ]:
# Plot modulo unit cell, gives an idea
plot_R_probes(jnp.remainder(R_probes_org, r), r)
plot_R_probes(jnp.remainder(R_probes, r), r, label=r"$R_{probe,opt}$")
plt.legend()

In [ ]:
def run_probes(R_probes, N_iter=400):
    probes = create_probes(R_probes, f_ms, N_photons, cdtype)
    ys = A(x_obj * probes)
    x_est = pie(
        ys,
        probes,
        alpha=0.5,
        gamma=0.75,
        N_iter=N_iter,
        obj_support=obj_support,
        A=A,
        A_dagger=A_dagger,
    )
    compare_estimate(x_est * probed, x_obj * probed, N_compare_crop=N_crop)


def relative_l2_error(x_est, x_true, eps=1e-12):
    diff = x_est - x_true
    return jnp.sum(jnp.abs(diff) ** 2) / (jnp.sum(jnp.abs(x_true) ** 2) + eps)


def compare_estimate_num(x_est_raw, x_obj, N_compare_crop=N_compare_crop):
    # Break phase and if applicable twin symmetry
    x_est = choose_representative(x_est_raw, x_obj)
    # Compensate seemingly common global amplitude scale mismatches
    obj_crop, _ = crop_and_pad(x_obj, N_compare_crop, 0)
    est_crop, _ = crop_and_pad(x_est, N_compare_crop, 0)

    est_crop_norm = normalize_image_intensity(est_crop)
    obj_crop_norm = normalize_image_intensity(obj_crop)
    return relative_l2_error(est_crop_norm, obj_crop_norm)


def run_probes_num(R_probes, N_iter=400):
    probes = create_probes(R_probes, f_ms, N_photons, cdtype)
    ys = A(x_obj * probes)
    x_est_raw = pie(
        ys,
        probes,
        alpha=0.5,
        gamma=0.75,
        N_iter=N_iter,
        obj_support=obj_support,
        A=A,
        A_dagger=A_dagger,
    )

    return compare_estimate_num(x_est_raw, x_obj)

In [ ]:
run_probes(R_probes)

In [ ]:
run_probes_num(R_probes, N_iter=1000)

In [ ]:
S = 100
# keys = jax.random.split(base_key, S)

deltas_samples = jax.random.uniform(
    base_key,
    shape=(S, M, 2),
    minval=-delta_r,
    maxval=delta_r,
)

errs = []
for s in tqdm(range(S)):
    err = run_probes_num(deltas_samples[s, :], N_iter=1000)
    errs.append(err)
plt.hist(errs)

In [ ]:
S = 100

deltas_samples = jax.random.uniform(
    base_key,
    shape=(S, M, 2),
    minval=-delta_r,
    maxval=delta_r,
)

deltas_samples_opt = np.zeros_like(deltas_samples)

print("Running reconstructions")
errs_basic = []
for s in tqdm(range(S)):
    err = run_probes_num(deltas_samples[s, :])
    errs_basic.append(err)

print("Optimizing sample positions")
for s in tqdm(range(S)):
    deltas_samples_opt[s, :] = optimize_for_uniformity(deltas_samples[s, :])


print("Running reconstructions with optimized probes")
errs_opt = []
for s in tqdm(range(S)):
    err = run_probes_num(deltas_samples_opt[s, :])
    errs_opt.append(err)

In [ ]:
plt.hist(errs_basic)
plt.show()
plt.hist(errs_opt)
plt.show()
np.mean(errs_basic), np.mean(errs_opt), np.std(errs_basic, ddof=1), np.std(
    errs_opt, ddof=1
), np.median(errs_basic), np.median(errs_opt)

In [ ]:
import jax.numpy as jnp
from scipy.stats import qmc


def create_R_probes_Sobol(S, M, delta_r, seed=1776):
    """
    Generate S independent Sobol-scrambled sets of M 2D offsets.

    M must be a power of 2 so that each sample uses Sobol's random_base2,
    preserving the Sobol balance/uniformity property per sample.
    """
    if M <= 0 or (M & (M - 1)) != 0:
        raise ValueError("M must be a positive power of 2.")

    m = int(np.log2(M))

    deltas = np.empty((S, M, 2), dtype=np.float32)

    for s in range(S):
        sampler = qmc.Sobol(d=2, scramble=True, seed=seed + s)
        u = sampler.random_base2(m=m)  # (M, 2), balanced Sobol set

        deltas[s] = 2.0 * delta_r * u - delta_r

    return jnp.asarray(deltas)


# M = 32
deltas_samples_sobol = create_R_probes_Sobol(S, M, delta_r)

In [ ]:
# plot_R_probes(R_probes_org,r)
# plot_R_probes(deltas_samples_sobol[0,:],r,label=r"$R_{probe,opt}$")

# Plot modulo unit cell, gives an idea
plot_R_probes(jnp.remainder(R_probes_org[:M], r), r)
plot_R_probes(
    jnp.remainder(deltas_samples_sobol[0, :M], r), r, label=r"$R_{probe,sobol}$"
)
plt.legend()
# run_probes()
probes = create_probes(deltas_samples_sobol[0, :M], f_ms, N_photons, cdtype=cdtype)
analyse_probe(probes, x_obj, r_extent)

In [ ]:
print("Running reconstructions with Sobel points")
errs_sobol = []
for s in tqdm(range(S)):
    err = run_probes_num(deltas_samples_sobol[s, :])
    errs_sobol.append(err)

In [ ]:
plt.hist(errs_sobol)
np.mean(errs_sobol), np.std(errs_sobol, ddof=1)

In [ ]:
print("Optimizing Sobol probe positions")
deltas_samples_sobol_opt = np.zeros_like(deltas_samples_sobol)

for s in tqdm(range(S)):
    deltas_samples_sobol_opt[s, :] = optimize_for_uniformity(
        deltas_samples_sobol[s, :], N_iter=50
    )


print("Running reconstructions with optimized probes")
errs_sobol_opt = []
for s in tqdm(range(S)):
    err = run_probes_num(deltas_samples_sobol_opt[s, :])
    errs_sobol_opt.append(err)

In [ ]:
plt.hist(errs_sobol)
plt.show()
plt.hist(errs_sobol_opt)
plt.show()
np.mean(errs_sobol), np.mean(errs_sobol_opt), np.std(errs_sobol, ddof=1), np.std(
    errs_sobol_opt, ddof=1
), np.median(errs_sobol), np.median(errs_sobol_opt)